In [7]:
import os
import requests
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "scripts"))
from get_token import get_siar_token

In [18]:

import urllib

base_url = "https://servicio.mapa.gob.es/siarapi" 
user = "48644221A" 
password = "968796876aA" 

# URLs 
url_user = f"{base_url}/API/V1/Autenticacion/cifrarCadena?cadena={user}" 
url_password = f"{base_url}/API/V1/Autenticacion/cifrarCadena?cadena={password}" 

# Peticiones 
resp_user = requests.get(url_user) 
resp_pass = requests.get(url_password) 
user_enc = urllib.parse.quote(resp_user.text.strip()) 
pass_enc = urllib.parse.quote(resp_pass.text.strip()) 

url_token = f"{base_url}/API/V1/Autenticacion/obtenerToken?Usuario={user_enc}&Password={pass_enc}" 

token = requests.get(url_token) 

print(token.status_code) 
token = token.text

200


In [9]:
# base URL del servicio SIAR
BASE = "https://servicio.mapa.gob.es/siarapi"

## INFO DATOS

In [33]:
def get_info_df(token: str, tipo: str) -> pd.DataFrame:
    """
    Consulta Info/{tipo} en SIAR y devuelve un DataFrame con el campo 'datos'.
    tipo: CCAA, PROVINCIAS, ESTACIONES, ACCESOS, CODIGOSVALIDACION
    """
    tipo = tipo.upper().strip()

    r = requests.get(
        f"{BASE}/API/V1/Info/{tipo}",
        params={"token": token},
        timeout=30
    )
    r.raise_for_status()

    payload = r.json()

    # Si la API devuelve MensajeRespuesta (error lógico)
    if payload.get("MensajeRespuesta"):
        raise RuntimeError(payload["MensajeRespuesta"])

    datos = payload.get("datos")

    # Normalmente 'datos' es lista de dicts -> DataFrame directo
    if isinstance(datos, list):
        return pd.DataFrame(datos)

    # Por si en algún caso viene como dict (no habitual en Info, pero por robustez)
    if isinstance(datos, dict):
        return pd.DataFrame([datos])

    # Si viene vacío o inesperado, devuelve DF vacío
    return pd.DataFrame()

In [34]:
# CCAA
df_ccaa = get_info_df(token, "CCAA")
df_ccaa.head()

,CCAA,Codigo
0,Andalucía,AND
1,Aragón,ARA
2,Principado de Asturias,AST
3,Islas Baleares,BAL
4,Castilla y León,CAL


In [23]:
# PROVINCIAS
df_provincias = get_info_df(token, "PROVINCIAS")
df_provincias.head()

,Provincia,Codigo,Codigo_CCAA
0,Álava,VI,PVA
1,Albacete,AB,CAM
2,Alicante/Alacant,A,VAL
3,Almería,AL,AND
4,Ávila,AV,CAL


In [15]:
# ESTACIONES
df_estaciones = get_info_df(token, "ESTACIONES")
df_estaciones

,Estacion,Codigo,Termino,Longitud,Latitud,Altitud,XUTM,YUTM,Huso,Fecha_Instalacion,Fecha_Baja,Red_Estacion
0,Tarazona,AB01,Tarazona de la Mancha,015512000W,391520000N,722,593160.00,4345720.00,30,1999-10-12T22:00:00,2025-06-16T22:00:00,Red de estaciones del Ministerio
1,Caudete,AB10,Caudete,005847000W,384404000N,563,675585.00,4289270.00,30,2000-12-10T23:00:00,None,Red de estaciones del Ministerio
2,Hellín,AB11,Hellín,013939625W,382752800N,498,616816.77,4258221.98,30,2025-06-03T22:00:00,None,Red de estaciones del Ministerio
3,Tarazona de la Mancha,AB12,Tarazona de la Mancha,015533101W,391410805N,721,592702.42,4343553.11,30,2025-06-16T22:00:00,None,Red de estaciones del Ministerio
4,Juanaco,AB02,Villarrobledo,023925000W,391558000N,705,529582.00,4346370.00,30,1999-10-12T22:00:00,None,Red de estaciones del Ministerio
...,...,...,...,...,...,...,...,...,...,...,...,...
628,Épila,Z05,Épila,011655000W,413459000N,327,643204.00,4604930.00,30,2003-09-17T22:00:00,None,Red de estaciones del Ministerio
629,Ejea de los Caballeros,Z06,Ejea de los Caballeros,011146000W,420551000N,316,649166.00,4662200.00,30,2003-09-22T22:00:00,None,Red de estaciones del Ministerio
630,Sádaba,Z07,Sádaba,011833000W,421602000N,432,639427.00,4680840.00,30,2003-09-22T22:00:00,None,Red de estaciones del Ministerio
631,Luna,Z08,Luna,005609000W,420544000N,406,670687.00,4662470.00,30,2003-09-23T22:00:00,None,Red de estaciones del Ministerio


In [21]:
# ACCESOS
df_accesos = get_info_df(token, "ACCESOS")
df_accesos.head()

,NumAccesosMinutoActual,MaxAccesosMinuto,NumAccesosDiaActual,MaxAccesosDia,RegistrosAcumuladosMinuto,MaxRegistrosMinuto,RegistrosAcumuladosDia,MaxRegistrosDia
0,3,100,17,1000,104,100,2394,10000


In [22]:
# CODIGOSVALIDACION
df_codigos_validadion = get_info_df(token, "CODIGOSVALIDACION")
df_codigos_validadion.head()

,Descripcion,IdCodigoValidacion
0,Dato incorporado sin ningún error (Nivel 1),100
1,El dato ha sido editado manualmente por un usu...,101
2,El dato ha sido validado en Nivel 1 y es válid...,103
3,El dato ha sido editado (Nivel1) y es válido ...,104
4,El dato ha sido insertado (Nivel1) y es válid...,105


## SERVICIOS DE DATOS

In [48]:
def get_max_date_for_station(token: str, station_id: str) -> str:
    """
    Devuelve la fecha máxima disponible (YYYY-MM-DD) para una estación.
    Sustituye ENDPOINT según el manual.
    """
    # EJEMPLO: cambia esto por el endpoint real del PDF
    # endpoint = f"{BASE}/API/V1/Datos/FechasDisponibles/ESTACION"
    # params = {"token": token, "Id": station_id}
    # r = requests.get(endpoint, params=params, timeout=30)

    raise NotImplementedError("Rellena el endpoint real del manual.")

def get_daily_data_latest(token: str, station_ids: list[str], datos_calculados: bool = True):
    # 1) toma una estación “referencia” para sacar max date
    max_date = get_max_date_for_station(token, station_ids[0])

    params = {
        "token": token,
        "Id": station_ids,
        "FechaInicial": max_date,
        "FechaFinal": max_date,
        "DatosCalculados": "true" if datos_calculados else "false",
    }

    r = requests.get(f"{BASE}/API/V1/Datos/Diarios/ESTACION", params=params, timeout=60)
    r.raise_for_status()
    return r.json()

In [50]:
r.text

'{"datos":[{"Fecha":"2025-11-01T00:00:00","Estacion":"HU01","TempMedia":15.090,"TempMax":22.260,"HorMinTempMax":1300,"TempMin":8.690,"HorMinTempMin":650,"HumedadMedia":69.390,"HumedadMax":100.000,"HorMinHumMax":720,"humedadMin":42.760,"HorMinHumMin":1310,"VelViento":0.873,"DirViento":347.500,"VelVientoMax":5.772,"HorMinVelMax":1235,"DirVientoVelMax":292.400,"Radiacion":11.150,"Precipitacion":0.000,"TempSuelo1":null,"TempSuelo2":null,"EtPMon":1.524012,"PePMon":0.0,"CodTempSuelo1":null,"CodTempSuelo2":null},{"Fecha":"2025-11-02T00:00:00","Estacion":"HU01","TempMedia":14.140,"TempMax":18.710,"HorMinTempMax":1210,"TempMin":9.950,"HorMinTempMin":2340,"HumedadMedia":60.800,"HumedadMax":83.100,"HorMinHumMax":650,"humedadMin":37.690,"HorMinHumMin":1340,"VelViento":3.894,"DirViento":280.200,"VelVientoMax":10.790,"HorMinVelMax":1502,"DirVientoVelMax":272.900,"Radiacion":11.080,"Precipitacion":0.000,"TempSuelo1":null,"TempSuelo2":null,"EtPMon":2.926253,"PePMon":0.0,"CodTempSuelo1":null,"CodTempSu